In [1]:
import sys
import subprocess
import time

!pip uninstall -y numpy bitsandbytes triton pandas scipy scikit-learn
!pip install -q numpy==1.26.4
!pip install -q pandas scipy scikit-learn tabulate
!pip install -q datasets==2.20.0 transformers==4.44.2 accelerate==0.33.0 bitsandbytes==0.45.5 triton==3.1.0 tqdm regex==2024.7.24

print("Installed dependencies. Restart the environment")

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bokeh 3.8.2 requires pandas>=1.2, which is not installed.
mlxtend 0.23.4 requires pandas>=0.24.2, which is

In [1]:
import os
import re
import gc
import json
import random
import sqlite3
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("GPU detected:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

def hard_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive/spider/spider"
DEV_JSON_PATH = os.path.join(BASE_DIR, "dev.json")
DATABASE_DIR = os.path.join(BASE_DIR, "database")

assert os.path.exists(DEV_JSON_PATH), "No dev.json"
assert os.path.exists(DATABASE_DIR), "No database/"

with open(DEV_JSON_PATH, "r", encoding="utf-8") as f:
    spider_data = json.load(f)

SELECTED_DBS = ["concert_singer", "pets_1"]
subset = [ex for ex in spider_data if ex["db_id"] in SELECTED_DBS]
selected_examples = subset
print(f"{len(selected_examples)} examples to evaluate")

GPU detected: Tesla T4
Mounted at /content/drive
87 examples to evaluate


In [2]:
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {MODEL_ID} on 4-bits...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.eval()
print("Model loaded successfully")

@torch.inference_mode()
def generate_text(prompt: str, max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text_input, return_tensors="pt", max_length=3072, truncation=True).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated_ids = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading Qwen/Qwen2.5-Coder-7B-Instruct on 4-bits...


...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully


In [3]:
def connect_to_db(db_id: str):
    db_path = os.path.join(DATABASE_DIR, db_id, f"{db_id}.sqlite")
    return sqlite3.connect(db_path)

def execute_sql(conn, sql: str) -> Dict[str, Any]:
    try:
        cursor = conn.cursor()
        cursor.execute(sql)
        return {"success": True, "rows": cursor.fetchall(), "error": None}
    except Exception as e:
        return {"success": False, "rows": [], "error": str(e)}

def get_table_names(conn) -> List[str]:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    return [row[0] for row in cursor.fetchall()]

def extract_relevant_tables_from_sql(sql: str, all_tables: List[str]) -> List[str]:
    sql_lower = sql.lower()
    return [t for t in all_tables if re.search(r"\b" + re.escape(t.lower()) + r"\b", sql_lower)] or all_tables

def normalize_cell(value: Any) -> str:
    if value is None: return ""
    val_str = str(value).strip().lower()
    val_str = re.sub(r"\s+", " ", val_str)
    try:
        num = float(val_str)
        return str(int(num)) if num.is_integer() else str(round(num, 6))
    except:
        return val_str

def normalize_result(rows: Any) -> List[Tuple[str, ...]]:
    if not rows: return []
    norm = []
    for row in rows:
        if not isinstance(row, (list, tuple)): row = [row]
        norm.append(tuple(normalize_cell(c) for c in row))
    return sorted(norm)

def flatten_cells(rows) -> List[str]:
    return [cell for row in rows for cell in row]

def evaluate_prediction(predicted, ground_truth) -> Dict[str, float]:
    pred_cells, gt_cells = flatten_cells(predicted), flatten_cells(ground_truth)

    if not pred_cells and not gt_cells:
        p = 1.0
    elif not pred_cells:
        p = 0.0
    else:
        gt_copy = list(gt_cells)
        correct = sum(1 for c in pred_cells if c in gt_copy and not gt_copy.remove(c))
        p = correct / len(pred_cells)

    if not gt_cells: r = 1.0 if not pred_cells else 0.0
    else:
        pred_copy = list(pred_cells)
        correct = sum(1 for c in gt_cells if c in pred_copy and not pred_copy.remove(c))
        r = correct / len(gt_cells)

    c = min(len(predicted), len(ground_truth)) / max(len(predicted), len(ground_truth)) if ground_truth or predicted else 1.0
    if not ground_truth and not predicted: c = 1.0
    elif not ground_truth or not predicted: c = 0.0

    return {"cell_precision": p, "cell_recall": r, "tuple_cardinality": c}

In [4]:
def get_schema_text(conn, relevant_tables: List[str]) -> str:
    lines = []
    for table in relevant_tables:
        cursor = conn.cursor()
        cursor.execute(f"PRAGMA table_info('{table}')")
        cols = [f"{col[1]} ({col[2]})" for col in cursor.fetchall()]
        lines.append(f"Table: {table}\nColumns: {', '.join(cols)}\n")
    return "\n".join(lines)

def pipeline_text_to_sql(example: Dict[str, Any]) -> Dict[str, Any]:
    conn = connect_to_db(example["db_id"])
    rel_tables = extract_relevant_tables_from_sql(example["query"], get_table_names(conn))
    schema_text = get_schema_text(conn, rel_tables)

    prompt = f"""You are an expert SQL generator. Write a valid SQLite query to answer the question based on the schema.

Schema:
{schema_text}

Question: {example['question']}

Return ONLY the SQL query. No explanations. No markdown formatting like ```sql."""

    raw_output = generate_text(prompt, max_new_tokens=256)

    sql = raw_output.replace("```sql", "").replace("```", "").strip()
    if ";" in sql: sql = sql[:sql.find(";") + 1]

    execution = execute_sql(conn, sql)
    conn.close()

    return {
        "generated_sql": sql,
        "success": execution["success"],
        "normalized_rows": normalize_result(execution["rows"]),
        "error": execution["error"]
    }

In [5]:
def serialize_database_markdown(conn, relevant_tables: List[str], max_rows: int = 30) -> str:
    serialized = []
    for table in relevant_tables:
        df = pd.read_sql_query(f"SELECT * FROM '{table}' LIMIT {max_rows}", conn)
        serialized.append(f"### Table: {table}\n{df.to_markdown(index=False)}")
    return "\n\n".join(serialized)

def pipeline_table_qa(example: Dict[str, Any]) -> Dict[str, Any]:
    conn = connect_to_db(example["db_id"])
    rel_tables = extract_relevant_tables_from_sql(example["query"], get_table_names(conn))
    serialized_tables = serialize_database_markdown(conn, rel_tables, max_rows=30)
    conn.close()

    prompt = f"""You are a data extraction system. Answer the user's question using ONLY the data in the tables provided below.

{serialized_tables}

Question: {example['question']}

You MUST output ONLY a valid JSON object with the exact format below. Do NOT add any extra text, explanations, or markdown blocks outside the JSON.
Format required:
{{
  "answer": [
    ["value_col1", "value_col2"],
    ["value_col1", "value_col2"]
  ]
}}
If the answer is a single value, format it as: {{"answer": [["value"]]}}.
If no answer is found, return {{"answer": []}}.
"""

    raw_output = generate_text(prompt, max_new_tokens=512)

    parsed_answer = []
    success = False
    error = None

    try:
        match = re.search(r"\{.*\}", raw_output.replace('\n', ' '), re.IGNORECASE)
        json_str = match.group(0) if match else raw_output

        data = json.loads(json_str)
        parsed_answer = data.get("answer", [])
        if isinstance(parsed_answer, list):
            success = True
        else:
            error = "JSON did not contain a valid list in 'answer' key."
            parsed_answer = []
    except Exception as e:
        error = str(e)

    return {
        "raw_model_output": raw_output,
        "parse_success": success,
        "normalized_rows": normalize_result(parsed_answer),
        "error": error
    }

In [6]:
def get_ground_truth(example: Dict[str, Any]) -> Dict[str, Any]:
    conn = connect_to_db(example["db_id"])
    exec_gt = execute_sql(conn, example["query"])
    conn.close()
    return normalize_result(exec_gt["rows"])

all_results = []
print("Pipeline evaluation...")

for idx, example in enumerate(tqdm(selected_examples)):
    gt_rows = get_ground_truth(example)

    sql_res = pipeline_text_to_sql(example)
    sql_metrics = evaluate_prediction(sql_res["normalized_rows"], gt_rows)

    qa_res = pipeline_table_qa(example)
    qa_metrics = evaluate_prediction(qa_res["normalized_rows"], gt_rows)

    row = {
        "id": idx,
        "db_id": example["db_id"],
        "question": example["question"],
        "gold_sql": example["query"],
        "ground_truth": gt_rows,

        "generated_sql": sql_res["generated_sql"],
        "sql_error": sql_res["error"],
        "sql_output": sql_res["normalized_rows"],
        "sql_precision": sql_metrics["cell_precision"],
        "sql_recall": sql_metrics["cell_recall"],
        "sql_cardinality": sql_metrics["tuple_cardinality"],

        "qa_raw_output": qa_res["raw_model_output"],
        "qa_error": qa_res["error"],
        "qa_output": qa_res["normalized_rows"],
        "qa_precision": qa_metrics["cell_precision"],
        "qa_recall": qa_metrics["cell_recall"],
        "qa_cardinality": qa_metrics["tuple_cardinality"],
    }
    all_results.append(row)

results_df = pd.DataFrame(all_results)
summary = pd.DataFrame([
    {
        "Method": "Text-to-SQL",
        "Cell Precision": results_df["sql_precision"].mean(),
        "Cell Recall": results_df["sql_recall"].mean(),
        "Tuple Cardinality": results_df["sql_cardinality"].mean(),
    },
    {
        "Method": "Direct Table-QA",
        "Cell Precision": results_df["qa_precision"].mean(),
        "Cell Recall": results_df["qa_recall"].mean(),
        "Tuple Cardinality": results_df["qa_cardinality"].mean(),
    }
])
print("\nAverage Metrics Summary:")
print(summary.to_markdown(index=False))

Pipeline evaluation...


  0%|          | 0/87 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation


Average Metrics Summary:
| Method          |   Cell Precision |   Cell Recall |   Tuple Cardinality |
|:----------------|-----------------:|--------------:|--------------------:|
| Text-to-SQL     |         0.963602 |       0.95977 |            0.981191 |
| Direct Table-QA |         0.600738 |       0.64589 |            0.689908 |


In [7]:
RUN_NAME = "qwen7b_coder_experiment"
OUTPUT_DIR = Path(f"/content/drive/MyDrive/spider_table_project_outputs/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(OUTPUT_DIR / "full_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)

sql_better = results_df[results_df["sql_recall"] > results_df["qa_recall"]]
qa_better = results_df[results_df["qa_recall"] > results_df["sql_recall"]]
both_fail = results_df[(results_df["sql_recall"] < 1.0) & (results_df["qa_recall"] < 1.0)]

sql_better.to_csv(OUTPUT_DIR / "sql_better_cases.csv", index=False)
qa_better.to_csv(OUTPUT_DIR / "qa_better_cases.csv", index=False)
both_fail.to_csv(OUTPUT_DIR / "both_fail_cases.csv", index=False)

report_md = f"""# Project: Text-to-SQL vs Direct Table QA

## Experimental Setup
- **Model:** {MODEL_ID}
- **Databases used:** {SELECTED_DBS}
- **Evaluated Questions:** {len(selected_examples)}

## Average Results
{summary.to_markdown(index=False)}

## Error Analysis Summary
- Cases where SQL was better: {len(sql_better)}
- Cases where QA was better: {len(qa_better)}
- Cases where both failed (Recall < 1): {len(both_fail)}
"""

with open(OUTPUT_DIR / "technical_report_base.md", "w") as f:
    f.write(report_md)

print(f"All files saved to: {OUTPUT_DIR}")

All files saved to: /content/drive/MyDrive/spider_table_project_outputs/qwen7b_coder_experiment
